# Enriquecimiento de base de papers con clasificaciones Scopus
Agrega columnas **ASJC Codes**, **Grupos Padre** y **Subgrupos** a la base de papers, cruzando por `Source Title` con el glosario de Scopus.

In [1]:
import pandas as pd
import re

## 1. Rutas de archivos

In [2]:
PAPERS_PATH  = r"C:\Users\RUTA DE LA BASE DE PAPERS"
SCOPUS_PATH  = r"C:\Users\RUTA DEL GLOSARIO DE SCOPUS.xlsx"
OUTPUT_PATH  = r"C:\Users\RUTA DONDE GUARDAR EL RESULTADO DEL CODIGO.xlsx"
SCOPUS_SHEET = "Scopus Sources Feb. 2026"

## 2. Cargar datos

In [3]:
df_papers = pd.read_excel(PAPERS_PATH, dtype=str)
print(f"Papers cargados: {len(df_papers)} filas")
df_papers.head(3)

Papers cargados: 13881 filas


,Authors,Author full names,Author(s) ID,Title,Year,Source Title,Volume,Issue,Art. No.,Page start,...,Publication Stage,Open Access,Source,EID,exactas_divisiones,DEPARTAMENTOS,INSTITUTOS,autores,autores con departamentos,autores con institutos
0,Minaberry Y.S.; Sagrera G.; García M.R.; Gutié...,"Minaberry, Yanina Susana (22634875800); Sagrer...",22634875800; 6506032464; 59757030100; 55954052...,Photostability and phototoxicity studies of a ...,2026,International Journal of Cosmetic Science,48,1,NaN,40,...,Final,NaN,Scopus,2-s2.0-105011957244,"Minaberry Y.S., D.Q.I.A.Q.F, Facultad de Cienc...",D.Q.I.A.Q.F,NaN,"Minaberry, Yanina Susana (22634875800); Svarc,...","Minaberry Y.S., D.Q.I.A.Q.F - D.Q.I.A.Q.F; Sva...",NaN
1,Bendersky M.; Rey R.D.,"Bendersky, Mariana (24450065700); Rey, Roberto...",24450065700; 7103291995,Motor Nerve Conduction of the Hypoglossal Nerv...,2026,Muscle and Nerve,73,3,NaN,490,...,Final,NaN,Scopus,2-s2.0-105027397055,"Bendersky M., Facultad de Medicina, Universida...",NaN,NaN,"Bendersky, Mariana (24450065700); Rey, Roberto...",NaN,NaN
2,Maskavizan A.J.; Dalibon E.L.; Prieto G.; Tuck...,"Maskavizan, A. Justina (58117862000); Dalibon,...",58117862000; 55305744800; 56097631300; 1024379...,Dry sliding wear resistance evaluation of cath...,2026,Surface and Coatings Technology,523,NaN,133202,NaN,...,Final,NaN,Scopus,2-s2.0-105028255213,"Márquez A.B., Universidad de Buenos Aires, Fac...",DEPTO FISICA,I.N.F.I.N.A,"Márquez, Adriana B. (7102678768)","Márquez A.B., Universidad de Buenos Aires - DE...","Márquez A.B., Universidad de Buenos Aires - I...."


In [4]:
df_scopus_raw = pd.read_excel(SCOPUS_PATH, sheet_name=SCOPUS_SHEET, dtype=str)
print(f"Scopus cargado: {len(df_scopus_raw)} filas, {len(df_scopus_raw.columns)} columnas")

Scopus cargado: 48192 filas, 52 columnas


## 3. Limpiar nombres de columnas del glosario Scopus
Las columnas pueden tener saltos de línea (`\n`) y espacios extra. Se normalizan en este paso.

In [5]:
def clean_col(name):
    """Reemplaza saltos de línea y espacios múltiples por un solo espacio."""
    return re.sub(r'\s+', ' ', str(name)).strip()

df_scopus = df_scopus_raw.copy()
df_scopus.columns = [clean_col(c) for c in df_scopus.columns]

print("Columnas del glosario Scopus (limpias):")
for c in df_scopus.columns:
    print(" -", repr(c))

Columnas del glosario Scopus (limpias):
 - 'Sourcerecord ID'
 - 'Source Title'
 - 'ISSN'
 - 'EISSN'
 - 'Active or Inactive'
 - 'Coverage'
 - 'Titles Discontinued by Scopus'
 - 'Article Language in Source (Three-Letter ISO Language Codes)'
 - 'Medline-sourced Title? (See additional details under separate tab.)'
 - 'Open Access Status'
 - 'Articles in Press Included?'
 - 'Added to List Feb. 2026'
 - 'Source Type'
 - 'Title History Indication'
 - 'Related Title 1'
 - 'Other Related Title 2'
 - 'Other Related Title 3'
 - 'Other Related Title 4'
 - 'Publisher'
 - 'Publisher Imprints Grouped to Main Publisher'
 - 'All Science Journal Classification Codes (ASJC)'
 - 'Top level: Life Sciences'
 - 'Top level: Social Sciences'
 - 'Top level: Physical Sciences'
 - 'Top level: Health Sciences'
 - '1000 General'
 - '1100 Agricultural and Biological Sciences'
 - '1200 Arts and Humanities'
 - '1300 Biochemistry, Genetics and Molecular Biology'
 - '1400 Business, Management and Accounting'
 - '1500 Ch

## 4. Definir columnas de ASJC, Grupos Padre y Subgrupos
Se buscan automáticamente los nombres reales en el archivo por coincidencia de palabras clave.

In [6]:
ASJC_COL = "All Science Journal Classification Codes (ASJC)"

PARENT_GROUPS = [
    "Top level: Life Sciences",
    "Top level: Social Sciences",
    "Top level: Physical Sciences",
    "Top level: Health Sciences",
]

SUBGROUPS = [
    "1100 Agricultural and Biological Sciences",
    "1200 Arts and Humanities",
    "1300 Biochemistry, Genetics and Molecular Biology",
    "1400 Business, Management and Accounting",
    "1500 Chemical Engineering",
    "1600 Chemistry",
    "1700 Computer Science",
    "1800 Decision Sciences",
    "1900 Earth and Planetary Sciences",
    "2000 Economics, Econometrics and Finance",
    "2100 Energy",
    "2200 Engineering",
    "2300 Environmental Science",
    "2400 Immunology and Microbiology",
    "2500 Materials Science",
    "2600 Mathematics",
    "2700 Medicine",
    "2800 Neuroscience",
    "2900 Nursing",
    "3000 Pharmacology, Toxicology and Pharmaceutics",
    "3100 Physics and Astronomy",
    "3200 Psychology",
    "3300 Social Sciences",
    "3400 Veterinary",
    "3500 Dentistry",
    "3600 Health Professions",
]

cols_disponibles = list(df_scopus.columns)

def find_best_match(expected, available_cols):
    """Coincidencia exacta primero, luego por normalización de espacios."""
    if expected in available_cols:
        return expected
    exp_norm = re.sub(r'\s+', ' ', expected).strip().lower()
    for col in available_cols:
        if re.sub(r'\s+', ' ', col).strip().lower() == exp_norm:
            return col
    return None

# Resolver columna ASJC
ASJC_COL_REAL = find_best_match(ASJC_COL, cols_disponibles)
if not ASJC_COL_REAL:
    candidates = [c for c in cols_disponibles if "ASJC" in c or "Classification" in c]
    ASJC_COL_REAL = candidates[0] if candidates else None
print(f"{'✅' if ASJC_COL_REAL else '❌'} ASJC → '{ASJC_COL_REAL}'")

# Resolver Grupos Padre
PARENT_GROUPS_REAL = []
for pg in PARENT_GROUPS:
    match = find_best_match(pg, cols_disponibles)
    if not match:
        keyword = pg.split(":")[-1].strip()
        cands = [c for c in cols_disponibles if keyword.lower() in c.lower()]
        match = cands[0] if cands else None
    if match:
        PARENT_GROUPS_REAL.append(match)
        print(f"  ✅ '{pg}' → '{match}'")
    else:
        print(f"  ❌ No encontrado: '{pg}'")

# Resolver Subgrupos
SUBGROUPS_REAL = []
for sg in SUBGROUPS:
    match = find_best_match(sg, cols_disponibles)
    if not match:
        code = sg[:4]
        cands = [c for c in cols_disponibles if c.startswith(code)]
        match = cands[0] if cands else None
    if match:
        SUBGROUPS_REAL.append(match)
    else:
        print(f"  ❌ No encontrado: '{sg}'")

print(f"\nGrupos Padre resueltos: {len(PARENT_GROUPS_REAL)}/{len(PARENT_GROUPS)}")
print(f"Subgrupos resueltos:    {len(SUBGROUPS_REAL)}/{len(SUBGROUPS)}")

✅ ASJC → 'All Science Journal Classification Codes (ASJC)'
  ✅ 'Top level: Life Sciences' → 'Top level: Life Sciences'
  ✅ 'Top level: Social Sciences' → 'Top level: Social Sciences'
  ✅ 'Top level: Physical Sciences' → 'Top level: Physical Sciences'
  ✅ 'Top level: Health Sciences' → 'Top level: Health Sciences'

Grupos Padre resueltos: 4/4
Subgrupos resueltos:    26/26


## 5. Normalizar títulos y construir tabla de lookup

In [7]:
def normalize(s):
    if pd.isna(s):
        return ""
    return re.sub(r'\s+', ' ', str(s)).strip().lower()

# Detectar columna de título en Scopus
SCOPUS_TITLE_COL = "Source Title"
if SCOPUS_TITLE_COL not in df_scopus.columns:
    candidates = [c for c in df_scopus.columns if "source" in c.lower() and "title" in c.lower()]
    SCOPUS_TITLE_COL = candidates[0] if candidates else SCOPUS_TITLE_COL
    print(f"🔄 Usando columna de título Scopus: '{SCOPUS_TITLE_COL}'")

df_scopus["_title_norm"] = df_scopus[SCOPUS_TITLE_COL].apply(normalize)

def get_labels(row, cols):
    labels = []
    for col in cols:
        if col in row.index:
            val = row[col]
            if pd.notna(val) and str(val).strip() != "":
                labels.append(str(val).strip())
    return " | ".join(labels) if labels else ""

# Construir lookup: title_norm -> {gp, sg, asjc}
lookup = {}
for _, row in df_scopus.iterrows():
    key = row["_title_norm"]
    if not key:
        continue
    gp   = get_labels(row, PARENT_GROUPS_REAL)
    sg   = get_labels(row, SUBGROUPS_REAL)
    asjc_raw = str(row[ASJC_COL_REAL]).strip() if ASJC_COL_REAL and pd.notna(row.get(ASJC_COL_REAL)) else ""

    if key not in lookup:
        lookup[key] = {"gp": set(), "sg": set(), "asjc": set()}
    if gp:
        lookup[key]["gp"].update(gp.split(" | "))
    if sg:
        lookup[key]["sg"].update(sg.split(" | "))
    if asjc_raw:
        # Los códigos ASJC pueden venir separados por espacios o punto y coma
        codes = re.split(r'[;\s]+', asjc_raw)
        lookup[key]["asjc"].update(c for c in codes if c and c != "nan")

print(f"Revistas únicas en lookup: {len(lookup)}")

Revistas únicas en lookup: 47967


## 6. Cruzar con la base de papers

In [11]:
PAPER_TITLE_COL = "Source Title"   # ← columna en tu base de papers

df_papers["_title_norm"] = df_papers[PAPER_TITLE_COL].apply(normalize)

def resolve(norm_title, field, sep=" | "):
    entry = lookup.get(norm_title)
    if entry is None:
        return ""
    return sep.join(sorted(entry[field]))

# ASJC: cambiar sep=" " por sep="; "
df_papers["All Science Journal Classification Codes (ASJC)"] = \
    df_papers["_title_norm"].apply(lambda t: resolve(t, "asjc", sep="; "))

# Grupos Padre y Subgrupos: el separador " | " ya está en resolve por defecto,
# cambiarlo a "; " en cada llamada
df_papers["Grupos Padre"] = df_papers["_title_norm"].apply(lambda t: resolve(t, "gp", sep="; "))
df_papers["Subgrupos"]    = df_papers["_title_norm"].apply(lambda t: resolve(t, "sg", sep="; "))

df_papers.drop(columns=["_title_norm"], inplace=True)

matched   = (df_papers["Grupos Padre"] != "").sum()
unmatched = (df_papers["Grupos Padre"] == "").sum()
print(f"✅ Filas con match:    {matched}")
print(f"❌ Filas sin match:    {unmatched}")
print(f"   Cobertura:          {matched/len(df_papers)*100:.1f}%")

✅ Filas con match:    12495
❌ Filas sin match:    1386
   Cobertura:          90.0%


## 7. (Opcional) Ver revistas sin match

In [9]:
sin_match = (
    df_papers.loc[df_papers["Grupos Padre"] == "", PAPER_TITLE_COL]
    .dropna().unique()
)
print(f"Revistas únicas sin match: {len(sin_match)}")
for r in sorted(sin_match)[:30]:
    print(" -", r)

Revistas únicas sin match: 405
 - 12th Latin American Conference on Learning Objects and Technologies, LACLO 2017
 - 14th International Workshops on Semantic Evaluation, SemEval 2020 - co-located 28th International Conference on Computational Linguistics, COLING 2020, Proceedings
 - 14th Marcel Grossman Meeting On Recent Developments in Theoretical and Experimental General Relativity, Astrophysics and Relativistic Field Theories, Proceedings
 - 15th Cologne-Twente Workshop on Graphs and Combinatorial Optimization, CTW 2017
 - 15th Marcel Grossmann Meeting on Recent Developments in Theoretical and Experimental General Relativity, Astrophysics, and Relativistic Field Theories, MG 2018
 - 19th International Conference on Harmonisation within Atmospheric Dispersion Modelling for Regulatory Purposes, Harmo 2019
 - 1st EAGE Conference on South Atlantic Offshore Energy Resources
 - 2017 11th International Congress on Engineered Material Platforms for Novel Wave Phenomena, Metamaterials 2017
 

## 8. Exportar resultado

In [12]:
df_papers.to_excel(OUTPUT_PATH, index=False)
print(f"Archivo guardado en: {OUTPUT_PATH}")
df_papers[[PAPER_TITLE_COL,
           "All Science Journal Classification Codes (ASJC)",
           "Grupos Padre",
           "Subgrupos"]].head(10)

Archivo guardado en: C:\Users\FCEyN\Desktop\unir\BASE ENRIQUECIDA.xlsx


,Source Title,All Science Journal Classification Codes (ASJC),Grupos Padre,Subgrupos
0,International Journal of Cosmetic Science,1302; 1505; 1601; 2708; 3002; 3003,Health Sciences; Life Sciences; Physical Sciences,"Biochemistry, Genetics and Molecular Biology; ..."
1,Muscle and Nerve,1314; 2728; 2737; 2804,Health Sciences; Life Sciences,"Biochemistry, Genetics and Molecular Biology; ..."
2,Surface and Coatings Technology,1600; 2505; 2508; 3104; 3110,Physical Sciences,Chemistry; Materials Science; Physics and Astr...
3,Ethology,1103; 1105,Life Sciences,Agricultural and Biological Sciences
4,Forum Mathematicum,2600; 2604,Physical Sciences,Mathematics
5,Folia Primatologica,1103; 1105,Life Sciences,Agricultural and Biological Sciences
6,European Physical Journal Plus,1507; 3100,Physical Sciences,Chemical Engineering; Physics and Astronomy
7,Plants,1105; 1110; 2303,Life Sciences; Physical Sciences,Agricultural and Biological Sciences; Environm...
8,Astronomy and Astrophysics,1912; 3103,Physical Sciences,Earth and Planetary Sciences; Physics and Astr...
9,Annals of Botany,1110,Life Sciences,Agricultural and Biological Sciences


In [15]:
# Cargar la hoja de códigos ASJC con su descripción
df_asjc_codes = pd.read_excel(SCOPUS_PATH, sheet_name="ASJC Classification Codes", dtype=str)

# Limpiar nombres de columna por las dudas
df_asjc_codes.columns = [clean_col(c) for c in df_asjc_codes.columns]

# Construir diccionario code -> description
asjc_dict = dict(zip(
    df_asjc_codes["Code"].str.strip(),
    df_asjc_codes["Description"].str.strip()
))

# Función que convierte una celda de códigos separados por "; " en sus nombres
def codes_to_names(codes_str):
    if not codes_str or pd.isna(codes_str):
        return ""
    codes = [c.strip() for c in str(codes_str).split(";") if c.strip()]
    names = [asjc_dict.get(c, f"Código no encontrado: {c}") for c in codes]
    return "; ".join(names)

df_papers["Campos de Estudio Específicos"] = \
    df_papers["All Science Journal Classification Codes (ASJC)"].apply(codes_to_names)

# Verificación rápida
df_papers[["Source Title",
           "All Science Journal Classification Codes (ASJC)",
           "Campos de Estudio Específicos"]].head(10)

,Source Title,All Science Journal Classification Codes (ASJC),Campos de Estudio Específicos
0,International Journal of Cosmetic Science,1302; 1505; 1601; 2708; 3002; 3003,Aging; Colloid and Surface Chemistry; Chemistr...
1,Muscle and Nerve,1314; 2728; 2737; 2804,Physiology; Neurology (clinical); Physiology (...
2,Surface and Coatings Technology,1600; 2505; 2508; 3104; 3110,General Chemistry; Materials Chemistry; Surfac...
3,Ethology,1103; 1105,"Animal Science and Zoology; Ecology, Evolution..."
4,Forum Mathematicum,2600; 2604,General Mathematics; Applied Mathematics
5,Folia Primatologica,1103; 1105,"Animal Science and Zoology; Ecology, Evolution..."
6,European Physical Journal Plus,1507; 3100,Fluid Flow and Transfer Processes; General Phy...
7,Plants,1105; 1110; 2303,"Ecology, Evolution, Behavior and Systematics; ..."
8,Astronomy and Astrophysics,1912; 3103,Space and Planetary Science; Astronomy and Ast...
9,Annals of Botany,1110,Plant Science


In [16]:
df_papers.to_excel(OUTPUT_PATH, index=False)
print(f"Archivo guardado en: {OUTPUT_PATH}")
df_papers[[PAPER_TITLE_COL,
           "All Science Journal Classification Codes (ASJC)",
           "Grupos Padre",
           "Subgrupos",
           "Campos de Estudio Específicos"]].head(10)

Archivo guardado en: C:\Users\FCEyN\Desktop\unir\BASE ENRIQUECIDA.xlsx


,Source Title,All Science Journal Classification Codes (ASJC),Grupos Padre,Subgrupos,Campos de Estudio Específicos
0,International Journal of Cosmetic Science,1302; 1505; 1601; 2708; 3002; 3003,Health Sciences; Life Sciences; Physical Sciences,"Biochemistry, Genetics and Molecular Biology; ...",Aging; Colloid and Surface Chemistry; Chemistr...
1,Muscle and Nerve,1314; 2728; 2737; 2804,Health Sciences; Life Sciences,"Biochemistry, Genetics and Molecular Biology; ...",Physiology; Neurology (clinical); Physiology (...
2,Surface and Coatings Technology,1600; 2505; 2508; 3104; 3110,Physical Sciences,Chemistry; Materials Science; Physics and Astr...,General Chemistry; Materials Chemistry; Surfac...
3,Ethology,1103; 1105,Life Sciences,Agricultural and Biological Sciences,"Animal Science and Zoology; Ecology, Evolution..."
4,Forum Mathematicum,2600; 2604,Physical Sciences,Mathematics,General Mathematics; Applied Mathematics
5,Folia Primatologica,1103; 1105,Life Sciences,Agricultural and Biological Sciences,"Animal Science and Zoology; Ecology, Evolution..."
6,European Physical Journal Plus,1507; 3100,Physical Sciences,Chemical Engineering; Physics and Astronomy,Fluid Flow and Transfer Processes; General Phy...
7,Plants,1105; 1110; 2303,Life Sciences; Physical Sciences,Agricultural and Biological Sciences; Environm...,"Ecology, Evolution, Behavior and Systematics; ..."
8,Astronomy and Astrophysics,1912; 3103,Physical Sciences,Earth and Planetary Sciences; Physics and Astr...,Space and Planetary Science; Astronomy and Ast...
9,Annals of Botany,1110,Life Sciences,Agricultural and Biological Sciences,Plant Science
